In [24]:
import gymnasium as gym
import numpy as np
import random
import time
from IPython.display import clear_output

In [25]:
class TicTacToeEnv(gym.Env):
    def __init__(self):
        super(TicTacToeEnv, self).__init__()
        self.action_space = gym.spaces.Discrete(9)
        self.observation_space = gym.spaces.Box(low=-1, high=1, shape=(9,), dtype=np.int8)
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.board = np.zeros(9, dtype=np.int8)
        self.current_player = 1
        self.done = False
        self.winner = None
        return self.board.copy(), {}

    def step(self, action):
        if self.done or self.board[action] != 0:
            return self.board.copy(), -10, True, False, {"reason": "Invalid move"}

        self.board[action] = self.current_player
        winner = self.check_winner()
        if winner == 1:
            self.done = True
            self.winner = 1
            return self.board.copy(), 10, True, False, {"reason": "Agent wins"}

        if self.is_board_full():
            self.done = True
            self.winner = 0
            return self.board.copy(), 0, True, False, {"reason": "Draw"}

        self.current_player = -1
        available_actions = [i for i in range(9) if self.board[i] == 0]
        if available_actions:
            opponent_action = random.choice(available_actions)
            self.board[opponent_action] = -1

            winner = self.check_winner()
            if winner == -1:
                self.done = True
                self.winner = -1
                return self.board.copy(), -10, True, False, {"reason": "Opponent wins"}

            if self.is_board_full():
                self.done = True
                self.winner = 0
                return self.board.copy(), 0, True, False, {"reason": "Draw"}

        self.current_player = 1
        return self.board.copy(), -1, False, False, {}

    def check_winner(self):
        win_patterns = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],
            [0, 3, 6], [1, 4, 7], [2, 5, 8],
            [0, 4, 8], [2, 4, 6]
        ]
        for pattern in win_patterns:
            if self.board[pattern[0]] == self.board[pattern[1]] == self.board[pattern[2]] != 0:
                return self.board[pattern[0]]
        return 0

    def is_board_full(self):
        return 0 not in self.board

    def render(self):
        symbols = {0: '.', 1: 'X', -1: 'O'}
        grid = [symbols[cell] for cell in self.board]
        print("\n".join([
            f" {grid[0]} | {grid[1]} | {grid[2]} ",
            "---+---+---",
            f" {grid[3]} | {grid[4]} | {grid[5]} ",
            "---+---+---",
            f" {grid[6]} | {grid[7]} | {grid[8]} "
        ]))
        print()

In [26]:
def train_agent(env, episodes=5000):
    Q = {}
    alpha = 0.1
    gamma = 0.9
    epsilon_start = 1.0
    epsilon_min = 0.01
    epsilon_decay = 0.999

    epsilon = epsilon_start

    wins = 0
    draws = 0
    losses = 0

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False

        while not done:
            s_key = tuple(state)

            if s_key not in Q:
                Q[s_key] = np.zeros(9, dtype=np.float32)

            available_actions = [i for i in range(9) if state[i] == 0]

            if not available_actions:
                break

            if random.uniform(0, 1) < epsilon:
                action = random.choice(available_actions)
            else:
                masked_q = np.where(state == 0, Q[s_key], -np.inf)
                action = np.argmax(masked_q)

            next_state, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated

            if done:
                Q[s_key][action] += alpha * (reward - Q[s_key][action])
            else:
                ns_key = tuple(next_state)
                if ns_key not in Q:
                    Q[ns_key] = np.zeros(9, dtype=np.float32)

                available_next_actions = [i for i in range(9) if next_state[i] == 0]
                if available_next_actions:
                    masked_q_next = np.where(next_state == 0, Q[ns_key], -np.inf)
                    best_next_action = np.argmax(masked_q_next)
                    td_target = reward + gamma * Q[ns_key][best_next_action]
                else:
                    td_target = reward
                td_error = td_target - Q[s_key][action]
                Q[s_key][action] += alpha * td_error

            state = next_state

        if env.winner == 1:
            wins += 1
        elif env.winner == 0:
            draws += 1
        elif env.winner == -1:
            losses += 1

        if epsilon > epsilon_min:
            epsilon *= epsilon_decay

        if episode % 1000 == 0 or episode == 4999:
            print(f"Эпизод {episode}, Epsilon: {epsilon:.4f}, Победы: {wins}, Ничьи: {draws}, Поражения: {losses}")

    return Q, wins, draws, losses

In [29]:
def run_and_visualize(env, q_table, games=3):
    for game in range(games):
        print(f"\n--- Демонстрация игры ---")
        state, _ = env.reset()
        done = False

        while not done:
            env.render()
            s_key = tuple(state)

            if s_key not in q_table:
                available_actions = [i for i in range(9) if state[i] == 0]
                action = random.choice(available_actions)
            else:
                masked_q = np.where(state == 0, q_table[s_key], -np.inf)
                action = np.argmax(masked_q)

            print(f"Игрок (X) выбирает клетку: {action}")
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            if done:
                env.render()
                if env.winner == 1:
                    print("Игрок (X) выиграл!")
                elif env.winner == -1:
                    print("Игрок (O) выиграл!")
                else:
                    print("Ничья!")
                break

            time.sleep(1)

In [31]:
env = TicTacToeEnv()
print("Обучение игрока в игре 'Крестики-нолики'")
q_table, wins, draws, losses = train_agent(env)

run_and_visualize(env, q_table, games=3)

Обучение игрока в игре 'Крестики-нолики'
Эпизод 0, Epsilon: 0.9990, Победы: 0, Ничьи: 1, Поражения: 0
Эпизод 1000, Epsilon: 0.3673, Победы: 595, Ничьи: 142, Поражения: 264
Эпизод 2000, Epsilon: 0.1351, Победы: 1371, Ничьи: 211, Поражения: 419
Эпизод 3000, Epsilon: 0.0497, Победы: 2280, Ничьи: 246, Поражения: 475
Эпизод 4000, Epsilon: 0.0183, Победы: 3225, Ничьи: 269, Поражения: 507
Эпизод 4999, Epsilon: 0.0100, Победы: 4187, Ничьи: 292, Поражения: 521

--- Демонстрация игры ---
 . | . | . 
---+---+---
 . | . | . 
---+---+---
 . | . | . 

Игрок (X) выбирает клетку: 0
 X | . | . 
---+---+---
 . | . | . 
---+---+---
 O | . | . 

Игрок (X) выбирает клетку: 1
 X | X | . 
---+---+---
 . | . | O 
---+---+---
 O | . | . 

Игрок (X) выбирает клетку: 2
 X | X | X 
---+---+---
 . | . | O 
---+---+---
 O | . | . 

Игрок (X) выиграл!

--- Демонстрация игры ---
 . | . | . 
---+---+---
 . | . | . 
---+---+---
 . | . | . 

Игрок (X) выбирает клетку: 0
 X | . | . 
---+---+---
 O | . | . 
---+---+---
 .